In [38]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Текущая папка:", Path.cwd())

Python: d:\ML\GitHub\.venv\Scripts\python.exe
Текущая папка: d:\ML\GitHub\findUNP


In [39]:
from pathlib import Path

# print(Path("find_unp_fixed.py").exists())
print(Path("find_unp_fixed_v2.py").exists())
print(Path("input.xlsx").exists())

True
True


In [51]:
# import find_unp_fixed
import find_unp_fixed_v2
print("Модуль успешно загружен")

Модуль успешно загружен


In [72]:
# names = find_unp_fixed.read_names("input.xlsx", column="ORG_NAME")
names = find_unp_fixed_v2.read_names("input.xlsx", name_column="ORG_NAME")

print("Количество организаций:", len(names))
print()
print("Первые 5 организаций:")

for item in names[:5]:
    print(item)

Количество организаций: 739

Первые 5 организаций:
(2, '5', '', '#нет#')
(3, '27', 'ЮЛ', 'ОАО "Могилевхлебопродукт"')
(4, '50', 'ЮЛ', 'ООО "ЕВРО')
(5, '51', 'ЮЛ', 'Филиал ОАО "Белагропромбанк"')
(6, '72', 'ЮЛ', 'ЗАО "Минский Транзитный Банк"')


In [ ]:
# first_name = names[0][1]
first_name = names[1][3]

print("Тестируем организацию:")
print(first_name)

In [ ]:
# Шаг 6. Выполняем тестовый запрос к ЕГР - дает ошибку если содержит кавычки
candidates = find_unp_fixed.egr_search(first_name, debug=True)

In [13]:
name1 = 'ОАО "Могилевхлебопродукт"'

# name2 = find_unp_fixed.normalize(name1)
name2 = find_unp_fixed_v2.normalize(name1)

print("Исходное название:")
print(name1)
print()
print("После normalize():")
print(name2)

Исходное название:
ОАО "Могилевхлебопродукт"

После normalize():
могилевхлебопродукт


In [14]:
# Шаг 8. Проверяем поиск нормализованного названия
# search_name = find_unp_fixed.normalize(first_name)
search_name = find_unp_fixed_v2.normalize(first_name)

print("Исходное:")
print(first_name)
print()
print("Для поиска:")
print(search_name)
print()
candidates = find_unp_fixed_v2.egr_search(search_name, debug=True)

Исходное:
ОАО "Могилевхлебопродукт"

Для поиска:
могилевхлебопродукт

URL: http://egr.gov.by/api/v2/egr/getShortInfoByRegName/%D0%BC%D0%BE%D0%B3%D0%B8%D0%BB%D0%B5%D0%B2%D1%85%D0%BB%D0%B5%D0%B1%D0%BE%D0%BF%D1%80%D0%BE%D0%B4%D1%83%D0%BA%D1%82
HTTP: 200
RAW: [{"ngrn":700099514,"dfrom":"1994-08-14T21:00:00.000+00:00","dto":"2025-02-27T21:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vnaim":"Открытое акционерное общество \"Могилевхлебопродукт\" - управляющая компания холдинга \"Могилевхлебопродукт\"","vn":"ОАО \"Могилевхлебопродукт\"- управляющая компания холдинга \"Могилевхлебопродукт\"","vfn":"Могилевхлебопродукт"}]
CANDIDATES: [
  {
    "unp": "700099514",
    "name": "Открытое акционерное общество \"Могилевхлебопродукт\" - управляющая компания холдинга \"Могилевхлебопродукт\"",
    "status": "Исключен из ЕГР",
    "raw": {
      "ngrn": 700099514,
      "dfrom": "1994-08-14T21:00:00.000+00:00",
      "dto": "2025-02-27T21:00:00.000+00:00",
      

In [ ]:
# Найдём потенциальные организационно-правовые формы автоматически.
from collections import Counter
import re

org_names = [name for _, name in names]

first_words = []

for name in org_names:
    words = re.findall(r'\S+', name)
    first_words.extend(words[:5])

counter = Counter(first_words)

print("Уникальных первых слов:", len(counter))
print()

for word, count in counter.most_common():
    print(f"{count:4}  {word}")

In [ ]:
# сколько строк из input.xlsx являются ИП и в каких формах они записаны.
import re

ip_names = []

for row_no, name in names:
    if re.search(r"^\s*ип\b", name, flags=re.IGNORECASE):
        ip_names.append((row_no, name))

print("Найдено строк с ИП:", len(ip_names))
print()

for row_no, name in ip_names[:100]:
    print(row_no, "|", name)

In [121]:
# Проверка наличия и доступности
from pathlib import Path

print(Path.cwd())
from pathlib import Path

for name in ["find_unp_fixed_v3.py", "input.xlsx"]:
    p = Path(name)
    print(name, "→", p.exists(), p.resolve())
import importlib
import find_unp_fixed_v3
importlib.reload(find_unp_fixed_v3)

d:\ML\GitHub\findUNP
find_unp_fixed_v3.py → True D:\ML\GitHub\findUNP\find_unp_fixed_v3.py
input.xlsx → True D:\ML\GitHub\findUNP\input.xlsx


<module 'find_unp_fixed_v3' from 'd:\\ML\\GitHub\\findUNP\\find_unp_fixed_v3.py'>

In [250]:
# Сначала посмотрим номера строк ЮЛ и ИП

records = find_unp_fixed_v3.read_names(
    "input.xlsx",
    id_column="id_r_raspost",
    type_column="TIP_ORG",
    name_column="ORG_NAME",
)

for row in records:
    row_no, record_id, tip_org, name = row
    
    if tip_org in ("ЮЛ", "ИП") and name and name != "#нет#":
        print(
            f"строка={row_no:>4} | "
            f"ID={record_id} | "
            f"{tip_org:2} | "
            f"{name}"
        )

строка=   2 | ID=27 | ЮЛ | ОАО "Могилевхлебопродукт"


In [155]:
# Запускаем тест одной строки ЮЛ
test_rows = find_unp_fixed_v3.test_single_record(
    records,
    row_number=3,
    cache_path="unp_cache.json",
)


=== ТЕСТ ОДНОЙ СТРОКИ ===
Строка Excel: 3
ID: 27
TIP_ORG: ЮЛ
ORG_NAME: ОАО "Могилевхлебопродукт"
Нормализованный поисковый запрос: могилевхлебопродукт
Источник результата: запрос ЕГР
URL: http://egr.gov.by/api/v2/egr/getShortInfoByRegName/%D0%BC%D0%BE%D0%B3%D0%B8%D0%BB%D0%B5%D0%B2%D1%85%D0%BB%D0%B5%D0%B1%D0%BE%D0%BF%D1%80%D0%BE%D0%B4%D1%83%D0%BA%D1%82
HTTP: 200
RAW: [{"ngrn":700099514,"dfrom":"1994-08-14T21:00:00.000+00:00","dto":"2025-02-27T21:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vnaim":"Открытое акционерное общество \"Могилевхлебопродукт\" - управляющая компания холдинга \"Могилевхлебопродукт\"","vn":"ОАО \"Могилевхлебопродукт\"- управляющая компания холдинга \"Могилевхлебопродукт\"","vfn":"Могилевхлебопродукт"}]
CANDIDATES: [
  {
    "unp": "700099514",
    "vfio": "",
    "vn": "ОАО \"Могилевхлебопродукт\"- управляющая компания холдинга \"Могилевхлебопродукт\"",
    "vnaim": "Открытое акционерное общество \"Могилевхлебопродукт\" - 

In [156]:
# Запускаем тест одной строки ЮЛ
test_rows = find_unp_fixed_v3.test_single_record(
    records,
    row_number=5,
    cache_path="unp_cache.json",
)


=== ТЕСТ ОДНОЙ СТРОКИ ===
Строка Excel: 5
ID: 51
TIP_ORG: ЮЛ
ORG_NAME: Филиал ОАО "Белагропромбанк"
Нормализованный поисковый запрос: филиал белагропромбанк
Источник результата: кэш

=== РЕЗУЛЬТАТ ===
УНП: —
Найденное название: —
Статус: —
Балл: 0.0
Решение: not_found

=== КАНДИДАТЫ ===
Кандидатов нет.


In [124]:
# Запускаем тест одной строки ИП
test_rows = find_unp_fixed_v3.test_single_record(
    records,
    row_number=711,
    cache_path="unp_cache.json",
)


=== ТЕСТ ОДНОЙ СТРОКИ ===
Строка Excel: 711
ID: 3264
TIP_ORG: ИП
ORG_NAME: ИП Вишнякова Т.В.
Нормализованный поисковый запрос: вишнякова
Источник результата: запрос ЕГР
URL: http://egr.gov.by/api/v2/egr/getShortInfoByRegName/%D0%B2%D0%B8%D1%88%D0%BD%D1%8F%D0%BA%D0%BE%D0%B2%D0%B0
HTTP: 200
RAW: [{"ngrn":400186004,"dfrom":"1993-02-10T22:00:00.000+00:00","dto":"1997-02-13T22:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Светлана Александровна"},{"ngrn":790017451,"dfrom":"2000-01-16T22:00:00.000+00:00","dto":"2003-01-20T22:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Валентина Дмитриевна"},{"ngrn":190078846,"dfrom":"2000-03-05T22:00:00.000+00:00","dto":"2001-05-30T21:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Ирина Николаевна"},{"ngrn":100746598,"dfrom":"1994-09-28T22:00:00.000+00:00","dto":"1997-02-24T22:00:00.00

In [125]:
import importlib
import find_unp_fixed_v3 as f

importlib.reload(f)

print(f.normalize("ИП Вишнякова Т.В.", "ИП"))
print(f.normalize('ОАО "Белагропромбанк"', "ЮЛ"))

names = find_unp_fixed_v3.read_names("input.xlsx", name_column="ORG_NAME")
for item in names[730:739]:
    print(item)
    
test_name = names[738][3]
test_tip = names[738][2]
print("Тестируем организацию:",test_name," Тип:",test_tip)
print(f.normalize(test_name, test_tip))


вишнякова
белагропромбанк
(732, '3285', 'ЮЛ', 'ЧТПУП "РенГлебАр"')
(733, '3286', 'ИП', 'Пашкевич С.Н.')
(734, '3287', 'ЮЛ', 'ООО "Бел-Изком"')
(735, '3288', 'ЮЛ', 'ООО "Стар Нью Фуд"')
(736, '3289', 'ИП', 'Рябов С.Ю.')
(737, '3290', 'ЮЛ', 'ООО "Сонит Маркет"')
(738, '3291', 'ИП', 'ИП Бураков')
(739, '3292', 'ИП', 'ИП Бураков')
(740, '5555', 'ИП', 'Ананенко Валерий Анатольевич')
Тестируем организацию: Ананенко Валерий Анатольевич  Тип: ИП
ананенко валерий анатольевич


In [246]:
# сначала исключить влияние кэша
from pathlib import Path

Path("unp_cache.json").unlink(missing_ok=True)
print("Кэш удалён")

Кэш удалён


In [255]:
# перезагрузить модупль:
import importlib
import find_unp_fixed_v3

importlib.reload(find_unp_fixed_v3)

<module 'find_unp_fixed_v3' from 'd:\\ML\\GitHub\\findUNP\\find_unp_fixed_v3.py'>

In [137]:
# не через test_single_record, а напрямую выполнить поиск:
print(find_unp_fixed_v3.normalize("Вишнякова Т.В.","ИП"))
candidates = find_unp_fixed_v3.egr_search("вишнякова",debug=True)
print("Количество кандидатов:", len(candidates))

for c in candidates[:10]:
    print(
        c.get("unp"),
        "|",
        c.get("vfio"),
        "|",
        c.get("status")
    )

вишнякова
URL: http://egr.gov.by/api/v2/egr/getShortInfoByRegName/%D0%B2%D0%B8%D1%88%D0%BD%D1%8F%D0%BA%D0%BE%D0%B2%D0%B0
HTTP: 200
RAW: [{"ngrn":400186004,"dfrom":"1993-02-10T22:00:00.000+00:00","dto":"1997-02-13T22:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Светлана Александровна"},{"ngrn":790017451,"dfrom":"2000-01-16T22:00:00.000+00:00","dto":"2003-01-20T22:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Валентина Дмитриевна"},{"ngrn":190078846,"dfrom":"2000-03-05T22:00:00.000+00:00","dto":"2001-05-30T21:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Ирина Николаевна"},{"ngrn":100746598,"dfrom":"1994-09-28T22:00:00.000+00:00","dto":"1997-02-24T22:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Нелли Николаевна"},{"ngrn":391927809,"dfrom":"2021-11-25T21:00:0

In [248]:
# import importlib
# import find_unp_fixed_v3

# importlib.reload(find_unp_fixed_v3)

candidates = find_unp_fixed_v3.egr_search(
    "Ананенко Валерий Анатольевич",
    debug=False
)

print("Количество кандидатов:", len(candidates))

matches = find_unp_fixed_v3.debug_ip_candidates(
    "Ананенко Валерий Анатольевич",
    candidates
)

Количество кандидатов: 1
ДИАГНОСТИКА КАНДИДАТОВ ИП
Исходное ФИО : Ананенко Валерий Анатольевич
Фамилия      : ананенко
Инициалы     : ва
Кандидатов   : 1

 1. УНП=700194255 | ФИО=Ананенко Валерий Анатольевич
    Фамилия: ананенко | Инициалы: ва | ФАМИЛИЯ + ИНИЦИАЛЫ | Статус: Действующий

--------------------------------------------------------------------------------
Совпадений фамилии + инициалов: 1
--------------------------------------------------------------------------------
1. УНП=700194255 | ФИО=Ананенко Валерий Анатольевич | Статус=Действующий


In [249]:
# import importlib
# import find_unp_fixed_v3

# importlib.reload(find_unp_fixed_v3)

test_rows = find_unp_fixed_v3.test_single_record(
    records,
    row_number=711,
    cache_path="unp_cache.json",
)


=== ТЕСТ ОДНОЙ СТРОКИ ===
Строка Excel: 711
ID: 3264
TIP_ORG: ИП
ORG_NAME: ИП Вишнякова Т.В.
Нормализованный поисковый запрос: вишнякова
Источник результата: запрос ЕГР
URL: http://egr.gov.by/api/v2/egr/getShortInfoByRegName/%D0%B2%D0%B8%D1%88%D0%BD%D1%8F%D0%BA%D0%BE%D0%B2%D0%B0
HTTP: 200
RAW: [{"ngrn":400186004,"dfrom":"1993-02-10T22:00:00.000+00:00","dto":"1997-02-13T22:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Светлана Александровна"},{"ngrn":790017451,"dfrom":"2000-01-16T22:00:00.000+00:00","dto":"2003-01-20T22:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Валентина Дмитриевна"},{"ngrn":190078846,"dfrom":"2000-03-05T22:00:00.000+00:00","dto":"2001-05-30T21:00:00.000+00:00","nsi00219":{"nsi00219":35961,"nksost":2,"vnsostk":"Исключен из ЕГР"},"vfio":"Вишнякова Ирина Николаевна"},{"ngrn":100746598,"dfrom":"1994-09-28T22:00:00.000+00:00","dto":"1997-02-24T22:00:00.00

In [178]:
# import importlib
# import find_unp_fixed_v3

# importlib.reload(find_unp_fixed_v3)

candidates = find_unp_fixed_v3.egr_search(
    "Вишнякова",
    debug=False
)

print("Количество кандидатов:", len(candidates))

matches = find_unp_fixed_v3.debug_ip_candidates(
    "Вишнякова Т.В.",
    candidates
)

Количество кандидатов: 74
ДИАГНОСТИКА КАНДИДАТОВ ИП
Исходное ФИО : Вишнякова Т.В.
Фамилия      : вишнякова
Инициалы     : тв
Кандидатов   : 74

 0. УНП=400186004 | ФИО=Вишнякова Светлана Александровна
    Фамилия: вишнякова | Инициалы: са | ТОЛЬКО ФАМИЛИЯ | Статус: Исключен из ЕГР
 0. УНП=790017451 | ФИО=Вишнякова Валентина Дмитриевна
    Фамилия: вишнякова | Инициалы: вд | ТОЛЬКО ФАМИЛИЯ | Статус: Исключен из ЕГР
 0. УНП=190078846 | ФИО=Вишнякова Ирина Николаевна
    Фамилия: вишнякова | Инициалы: ин | ТОЛЬКО ФАМИЛИЯ | Статус: Исключен из ЕГР
 0. УНП=100746598 | ФИО=Вишнякова Нелли Николаевна
    Фамилия: вишнякова | Инициалы: нн | ТОЛЬКО ФАМИЛИЯ | Статус: Исключен из ЕГР
 0. УНП=391927809 | ФИО=Вишнякова Светлана Анатольевна
    Фамилия: вишнякова | Инициалы: са | ТОЛЬКО ФАМИЛИЯ | Статус: Действующий
 0. УНП=193485217 | ФИО=Вишнякова Ольга Станиславовна
    Фамилия: вишнякова | Инициалы: ос | ТОЛЬКО ФАМИЛИЯ | Статус: Исключен из ЕГР
 0. УНП=192472326 | ФИО=Вишнякова Алеся Михайловна


In [238]:
# import importlib
# import find_unp_fixed_v3

importlib.reload(find_unp_fixed_v3)

test_rows = find_unp_fixed_v3.test_single_record(
    records,
    row_number=711,
    cache_path="unp_cache.json",
)


=== ТЕСТ ОДНОЙ СТРОКИ ===
Строка Excel: 711
ID: 3264
TIP_ORG: ИП
ORG_NAME: ИП Вишнякова Т.В.
Нормализованный поисковый запрос: вишнякова
Источник результата: кэш

=== РЕЗУЛЬТАТ ===
УНП: —
Найденное название: —
Статус: —
Балл: 100.0
Решение: manual_multiple

=== КАНДИДАТЫ ===
1. УНП=790768787 | Название=Вишнякова Татьяна Викторовна | Балл=100.0 | Статус=Действующий
2. УНП=790255498 | Название=Вишнякова Татьяна Викторовна | Балл=100.0 | Статус=Исключен из ЕГР
3. УНП=490374859 | Название=Вишнякова Татьяна Викторовна | Балл=100.0 | Статус=Действующий
4. УНП=700308319 | Название=Вишнякова Татьяна Васильевна | Балл=100.0 | Статус=Исключен из ЕГР


Программа пройдёт по всем строкам:
input.xlsx
   ↓
read_names()
   ↓
make_rows()
   ↓
best_match()
   ↓
result.xlsx

main() именно это и делает, если --test-row не указан.

In [262]:
!python find_unp_fixed_v3.py input.xlsx
# !python find_unp_fixed_v3.py input.xlsx > test_result.txt 2>&1

Будет обработано строк: 10

Готово: 5/10 найдено.
Пустых названий: 0
Результат: result.xlsx


In [ ]:
#Важно: это запускает именно find_unp_fixed_v3.py как самостоятельную программу, 
#поэтому в input.xlsx должен находиться ваш тестовый набор, 
#а пути к input.xlsx, result.xlsx, unp_cache.json и т. п. должны соответствовать тому, как они заданы в v3.

!powershell -Command "python find_unp_fixed_v3.py 2>&1 | Tee-Object -FilePath test_result.txt"

In [ ]:
with open("test_result.txt", "r", encoding="utf-8") as f:
    print(f.read())

 find_unp_fixed_v3_google.py
 Важное примечание по поводу конца функции mainПосле того как вы исправите make_rows, скрипт может выдать еще одну ошибку в самом конце функции main() (строки 984–985):

Здесь автор кода наоборот — пытается работать со строками результата row как с кортежами/списками по индексу [8], хотя функция make_rows возвращает список словарей (rows.append({"ID": ..., "Решение": ...})). Позиция 8 (девятый элемент) действительно соответствует ключу "Решение".



In [ ]:
found = sum(row[8] in ("auto", "review") for row in rows)
empty = sum(row[8] == "empty_name" for row in rows)

# на
found = sum(row.get("Решение") in ("auto", "review") for row in rows)
empty = sum(row.get("Решение") == "empty_name" for row in rows)


